A notebook to test the functions of the new vrdr_resampler Python module.

In [ ]:
# auto reload
%load_ext autoreload
%autoreload 2

In [ ]:
import vrdr_resampler as vrdr
from pathlib import Path

# Loading the CaSSIS Spectral Response

We will set the CaSSIS identifier as simply 'c', that will be appended to the start of products.

In [ ]:
inst_id = 'c'

In [ ]:
# instrument path
inst_path = Path('instrument_response', 'CaSSIS-2019.csv')

In [ ]:
cassis_data = vrdr.load_channel_profiles(inst_path)
band_arr, arr_wvls, band_list, band_cwls = vrdr.unpack_channel_profiles(cassis_data)

In [ ]:
fig, ax = vrdr.plot_channel_profiles(cassis_data, instrument='CaSSIS-2019')

# Loading a VRDR Tile

Setting the data directory:

In [ ]:
vrdr_path = Path('/Volumes', 'RS_STORE', 'vrdr')

Setting the output directory:

In [ ]:
output_path = vrdr_path.parent / 'vrdr_cassis'
output_path.mkdir(exist_ok=True) 

Finding the quadrant directories:

In [ ]:
mc_paths = vrdr.get_quadrant_dirs(vrdr_path)
mc_path = mc_paths[0]  # Example: process the first quadrant directory

Finding the tile paths in the quadrant directory:

In [ ]:
lbl_paths = vrdr.get_tile_lbls(mc_path)
lbl_path = lbl_paths[0]  # Example: process the first label file

Preparing the output file directory:

In [ ]:
out_dir = vrdr.prepare_output_dir(lbl_path, mc_path, output_path, overwrite=True)
out_dir

Loading the tile, getting the band centre-wavelengths, and the no-data value:

In [ ]:
vrdr_cube, vrdr_cwls, no_data = vrdr.load_vrdr_tile(lbl_path)

# Resampling to the Spectral Response to a CaSSIS I/F Cube

In [ ]:
cifcub = vrdr.resample_cube(band_arr, arr_wvls, vrdr_cube, vrdr_cwls, no_data)

# Exporting the I/F Cube

In [ ]:
out_file = vrdr.export_ifcub_geotiff(cifcub,band_list,band_cwls,no_data,inst_id,lbl_path,out_dir)
out_hdr = vrdr.write_ifcub_envi_hdr(band_list, band_cwls, no_data,[4,2,1], inst_id, lbl_path, out_dir)

## Verifying the Resampling Method

In [ ]:
b_w = arr_wvls
c_w = vrdr_cwls
b_a = band_arr
resampled_profiles = [vrdr.interp_band_profile(b_p, b_w, c_w) for b_p in b_a.T]

In [ ]:
import numpy as np
import pandas as pd
resampled_inst = pd.DataFrame(data=np.array(resampled_profiles).T, columns=band_list, index=c_w)

In [ ]:
fig, ax = vrdr.plot_channel_profiles(resampled_inst, instrument='CaSSIS-2019 Resampled')

Checking a random selection of VRDR tile spectra

In [ ]:
# select random locations in the last 2 dimensions of the cube
import random
import matplotlib.pyplot as plt

num_samples = 10
x_size, y_size = vrdr_cube.shape[1], vrdr_cube.shape[2]
random_coords = [(random.randint(0, x_size - 1), random.randint(0, y_size - 1)) for _ in range(num_samples)]

fig, ax = plt.subplots(figsize=(5, 4))

y_max = 0.4

for band in resampled_inst.columns:
    ax.plot(resampled_inst.index, y_max*resampled_inst[band]/np.nanmax(resampled_inst[band]), label=band, linestyle='-',marker='o', alpha=0.2, markersize=1)
    # fill the area under each band profile
    ax.fill_between(resampled_inst.index, 0, y_max*resampled_inst[band]/np.nanmax(resampled_inst[band]), alpha=0.1)

for x, y in random_coords:
    vrdr_spectrum = vrdr_cube[:, x, y]
    cassis_spectrum = cifcub[:, x, y]

    ax.plot(vrdr_cwls, vrdr_spectrum, label=f'{x,y}', marker='o', markersize=3, linestyle='None', alpha=0.5)
    # get colour used in the plot above
    color = ax.get_lines()[-1].get_color()
    ax.plot(band_cwls, cassis_spectrum, label=f'{x,y}', marker='x', markersize=6, color=color)
    ax.set_xlabel('Wavelength (nm)')
    ax.set_ylabel('Reflectance')
# plt.legend()
ax.set_title(f'Randomly selected pixel spectra')
ax.set_ylim(0, y_max)
ax.grid()
fig.show()

Note that data points with wavelengths <400 nm (i.e. the apparently noisy channels) are 0-weighted by the CaSSIS BLU transmission, so they effectively are not counted.

# Prepare Products

In [ ]:
cstcub, st_params = vrdr.stretch_cube(cifcub, no_data) # cstcub = cassis stretched cube

In [ ]:
cstcub_check, st_params_check = vrdr.stretch_cube(cifcub, no_data, st_params=st_params)  # verify that applying the same stretch parameters returns the same result

In [ ]:
st_params

In [ ]:
cifcub_mod = cassis_add_sblu(cifcub, no_data)
st_cub, st_params = vrdr.stretch_cube(cifcub_mod, no_data)
bp_id = cassis_browse_products[0]
bp_func = cassis_browse_product(bp_id)
bp, bp_bands = bp_func(st_cub)
plt.imshow(bp)
plt.title(f'CaSSIS {bp_id} Browse Product')
plt.show()

In [ ]:
import cassis_products as cp

In [ ]:
cifcub_mod = cp.cassis_add_sblu(cifcub, no_data)
st_cub, st_params = vrdr.stretch_cube(cifcub_mod, no_data)
for bp_id in cp.BP_LIST:
    bp_func = cp.cassis_browse_product(bp_id)
    bp, bp_bands = bp_func(st_cub)
    vrdr.export_bp_geotiff(bp, bp_id, 255, bp_bands, lbl_path, out_dir)
    plt.imshow(bp)
    plt.title(f'CaSSIS {bp_id} Browse Product')
    plt.show()

In [ ]:
for bp_id in ['CBRC1', 'CBRC2']:
    sup, sup_bands = vrdr.cassis_composite_sup(bp_id, cassis_cube, no_data)
    bp_stretch = vrdr.stretch_cube(sup, no_data).transpose(1, 2, 0)
    vrdr.export_bp_geotiff(bp_stretch, bp_id, 255, sup_bands, lbl_path, out_dir)
    plt.imshow(bp_stretch)
    plt.title(f'CaSSIS {bp_id} Browse Product')
    plt.show()

# Export the Cube and Summary Product Stack to GeoTIFF with ENVI Headers

In [ ]:
out_file = vrdr.export_cube_geotiff(cassis_cube,band_list,band_cwls,no_data,lbl_path,out_dir)
out_hdr = vrdr.write_cube_envi_hdr(band_list, band_cwls, no_data,[4,2,1], lbl_path, out_dir)

In [ ]:
sups_cube, sp_list = vrdr.sups_from_cube(cassis_cube, ['PBR','PNR','RPR','ICE'],no_data)
out_file = vrdr.export_sups_geotiff(sups_cube,sp_list,no_data,lbl_path,out_dir)
out_hdr = vrdr.write_sups_envi_hdr(sp_list, no_data, [3, 1, 2], lbl_path, out_dir)

# Making Quadrant Wide Mosaics

In [ ]:
import rasterio as rio
r = rio.open(cube_file)

In [ ]:
stats = r.statistics(1)

In [ ]:
stats.mean - 5*stats.std, stats.mean + 5*stats.std

In [ ]:
cube = r.read(1)

In [ ]:
cube[cube == no_data] = np.nan
np.nanpercentile(cube, 0.01), np.nanpercentile(cube, 99.95)

In [ ]:
# get cassis quadrant directory paths
vrdr_path = Path('/Volumes', 'RS_STORE', 'vrdr_cassis')
output_path = vrdr_path.parent / 'vrdr_cassis_quads'
output_path.mkdir(exist_ok=True) 

mc_paths = vrdr.get_quadrant_dirs(vrdr_path)

for mc_path in mc_paths:
    print('Processing MC Directory: ', mc_path.name)
    # get lbl files but ignore hidden files
    tile_paths = vrdr.get_tile_dirs(mc_path)
    # sort

    tile_stretch_list = []
    for tile_path in tile_paths:
        print(' - ', tile_path.stem)
        # get tiff files
        cube_file = [f for f in tile_path.glob('*ccube*.tif')][0]        
        stretch_list = vrdr.read_geotiff_stretch_stats(cube_file)
        tile_stretch_list.append(np.array(stretch_list))
    tile_stretch_array = np.array(tile_stretch_list)

    stretch_min = np.min(tile_stretch_array[:,:,0], axis=0)
    stretch_max = np.max(tile_stretch_array[:,:,1], axis=0)

    for tile_path in tile_paths:
        print(' - ', tile_path.stem)

        cube_file = [f for f in tile_path.glob('*ccube*.tif')][0]
        cube, no_data = vrdr.read_geotiff_cube(cube_file)
        gbl_stretch_cube = vrdr.stretch_cube(cube, no_data, stretch_min=stretch_min, stretch_max=stretch_max)
        npb_gbl, npb_list = vrdr.cassis_browse_product('NPB', cube, gbl_stretch_cube, no_data)

        # get the label path from the vrdr directory
        lbl_path = Path(mc_path.parent.parent, 'vrdr', mc_path.stem, f'{tile_path.stem}.lbl')

        out_path = Path(output_path, mc_path.stem)
        out_path.mkdir(parents=True, exist_ok=True)

        vrdr.export_bp_geotiff(npb_gbl, 'NPBGBL', 255, npb_list, lbl_path, tile_path)
        

# Running the Loop

In [ ]:
inst_path = Path('.', 'CaSSIS-2019.csv')
vrdr_path = Path('/Volumes', 'RS_STORE', 'vrdr')
output_path = vrdr_path.parent / 'vrdr_cassis'
output_path.mkdir(exist_ok=True) 

cassis_data = vrdr.load_channel_profiles(inst_path)
band_arr, arr_wvls, band_list, band_cwls = vrdr.unpack_channel_profiles(cassis_data)

mc_paths = vrdr.get_quadrant_dirs(vrdr_path)

for mc_path in mc_paths:
    if mc_path.stem not in ['mc16']:
        continue
    print('Processing MC Directory: ', mc_path.name)
    # get lbl files but ignore hidden files
    lbl_paths = vrdr.get_tile_lbls(mc_path)
    # sort
    
    for lbl_path in lbl_paths:
        print(' - ', lbl_path.stem)

        out_dir = vrdr.prepare_output_dir(lbl_path, mc_path, output_path, overwrite=False)
        if not out_dir:
            continue

        vrdr_cube, vrdr_wvls, no_data = vrdr.load_vrdr_tile(lbl_path)

        cassis_cube = vrdr.resample_cube(band_arr, arr_wvls, vrdr_cube, vrdr_wvls, no_data)

        cassis_stretch_cube = vrdr.stretch_cube(cassis_cube, no_data)

        for bp_id in ['NPB', 'NRB', 'NRP', 'RPB', 'RGB']:
            bp, bp_bands = vrdr.cassis_browse_product(bp_id, cassis_cube, cassis_stretch_cube, no_data)
            vrdr.export_bp_geotiff(bp, bp_id, 255, bp_bands, lbl_path, out_dir)

        for bp_id in ['CBRC1', 'CBRC2']:
            sup, sup_bands = vrdr.cassis_composite_sup(bp_id, cassis_cube, no_data)
            bp_stretch = vrdr.stretch_cube(sup, no_data).transpose(1, 2, 0)
            vrdr.export_bp_geotiff(bp_stretch, bp_id, 255, sup_bands, lbl_path, out_dir)

        out_file = vrdr.export_cube_geotiff(cassis_cube,band_list,band_cwls,no_data,lbl_path,out_dir)
        out_hdr = vrdr.write_cube_envi_hdr(band_list, band_cwls, no_data,[4,2,1], lbl_path, out_dir)

        sups_cube, sp_list = vrdr.sups_from_cube(cassis_cube, ['PBR','PNR','RPR','ICE'],no_data)
        out_file = vrdr.export_sups_geotiff(sups_cube,sp_list,no_data,lbl_path,out_dir)
        out_hdr = vrdr.write_sups_envi_hdr(sp_list, no_data, [3, 1, 2], lbl_path, out_dir)
        

Now I'd like a quick script that loops over all processed tiles in a quadrant, gets the quadrant-wide stretch parameters, and then makes a mosaic browse product at that level.